# 00 — Setup do ambiente

Criação da estrutura do Lakehouse no Unity Catalog: os três schemas da arquitetura medalhão
e o Volume que recebe os arquivos brutos da CVM.

Este notebook roda **uma única vez**, antes de todo o resto do pipeline. Os comandos usam
`IF NOT EXISTS`, portanto são idempotentes: executar de novo não altera nada.

**Estrutura criada**

```
workspace (catálogo)
├── bronze          dados como recebidos
│   └── raw_cvm     Volume com os arquivos originais (ZIP e CSV)
├── silver          dados limpos, tipados e modelados
└── gold            tabelas analíticas para as perguntas de negócio
```

As camadas foram implementadas como schemas dentro do catálogo `workspace`, e não como
catálogos separados, por limitação da Databricks Free Edition. A separação lógica e a
progressão bruto → limpo → pronto para consumo são preservadas.

## 1. Schemas das três camadas

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze
  COMMENT 'Camada bruta: dados da CVM como recebidos, sem transformação de conteúdo. Todos os campos em texto, com metadados de origem e data de ingestão';

CREATE SCHEMA IF NOT EXISTS workspace.silver
  COMMENT 'Camada limpa: tipagem correta, deduplicação, padronização e dimensões conformadas';

CREATE SCHEMA IF NOT EXISTS workspace.gold
  COMMENT 'Camada analítica: tabelas agregadas e modeladas para responder às perguntas de negócio';

## 2. Volume para os arquivos brutos

O Volume é o storage gerenciado do Unity Catalog onde ficam os arquivos originais baixados
do Portal de Dados Abertos da CVM. Fica sob o schema bronze porque é a origem física da
camada bruta.

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_cvm
  COMMENT 'Arquivos originais baixados do Portal de Dados Abertos da CVM (licença ODbL): informes diários mensais em ZIP e registro de fundos, classes e subclasses';

## 3. Conferência

Confirma que a estrutura foi criada antes de seguir para a ingestão.

In [ ]:
%sql
SHOW SCHEMAS IN workspace;

In [ ]:
%sql
SHOW VOLUMES IN workspace.bronze;

In [ ]:
# Arquivos presentes no Volume
import os
VOL = "/Volumes/workspace/bronze/raw_cvm"
if os.path.exists(VOL):
    for f in sorted(os.listdir(VOL)):
        print(f)
else:
    print("Volume ainda vazio. Faca upload dos arquivos da CVM pelo Catalog Explorer:")
    print("Catalog > workspace > bronze > Volumes > raw_cvm > Upload")

---

**Próximo passo:** upload dos arquivos da CVM para o Volume, pelo Catalog Explorer, e
execução do notebook `01_bronze_ingestao`.

Arquivos necessários:
- `inf_diario_fi_AAAAMM.zip` — informes diários (11 meses, de 09/2025 a 07/2026)
- `registro_fundo_classe.zip` — cadastro de fundos, classes e subclasses

Ambos disponíveis em https://dados.cvm.gov.br